In [1]:
# Here we are using the same Skeleton of chatbot and improving it by adding loops and conditions to store the chat history.
from langgraph.graph import StateGraph,START,END
from langchain_openai import ChatOpenAI
from typing import TypedDict, Literal
import operator
from typing import List,Annotated
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
load_dotenv()

llm=ChatOpenAI()

python-dotenv could not parse statement starting at line 5


In [3]:
class JokeState(TypedDict):
    topic:str
    joke:str
    explanation:str


In [4]:
def generate_joke(state:JokeState):
    prompt=f"generate a joke for the topic :{state['topic']}"
    result=llm.invoke(prompt).content
    return {'joke':result}

def generate_explanation(state:JokeState):
    prompt=f"generate a explanation for the joke :{state['joke']}"
    result=llm.invoke(prompt).content
    return {'explanation': result}


In [5]:
graph =StateGraph(JokeState)
graph.add_node('generate_joke',generate_joke)
graph.add_node('generate_explanation',generate_explanation)

graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','generate_explanation')
graph.add_edge('generate_explanation',END)

#now build the checkpointer object for InMemorySaver class 
checkpointer=InMemorySaver()


workflow=graph.compile(checkpointer=checkpointer)


#Now create a threadId for this conversation
config={'configurable':{'thread_id':'1'}} 
workflow.invoke({'topic':"pizza"},config=config)

{'topic': 'pizza',
 'joke': 'Why did the slice of pizza go to the party alone? Because it wanted to be the "big cheese" of the night!',
 'explanation': 'This joke plays on the idea that being the "big cheese" is a term used to describe someone who is important or in charge. In this case, the slice of pizza wants to be seen as the most important or significant guest at the party, hence why it goes alone. The humor comes from the playful twist on the term "big cheese" and the absurdity of a slice of pizza wanting to be the center of attention at a party.'}

In [6]:
workflow.get_state(config)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the slice of pizza go to the party alone? Because it wanted to be the "big cheese" of the night!', 'explanation': 'This joke plays on the idea that being the "big cheese" is a term used to describe someone who is important or in charge. In this case, the slice of pizza wants to be seen as the most important or significant guest at the party, hence why it goes alone. The humor comes from the playful twist on the term "big cheese" and the absurdity of a slice of pizza wanting to be the center of attention at a party.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19bf09-e6ae-697b-8002-ed73e54238cc'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-19T17:08:37.745906+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19bf09-d424-60a3-8001-a36faa231204'}}, tasks=(), interrupts=())

In [7]:
list(workflow.get_state_history(config)) # this will give all the intermediate state history of the graph

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the slice of pizza go to the party alone? Because it wanted to be the "big cheese" of the night!', 'explanation': 'This joke plays on the idea that being the "big cheese" is a term used to describe someone who is important or in charge. In this case, the slice of pizza wants to be seen as the most important or significant guest at the party, hence why it goes alone. The humor comes from the playful twist on the term "big cheese" and the absurdity of a slice of pizza wanting to be the center of attention at a party.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19bf09-e6ae-697b-8002-ed73e54238cc'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-19T17:08:37.745906+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19bf09-d424-60a3-8001-a36faa231204'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topi

In [8]:
config2={'configurable':{'thread_id':'2'}}
workflow.invoke({'topic':'Cricket'},config=config2)

{'topic': 'Cricket',
 'joke': 'Why did the cricket team go to the bakery?\n\nBecause they heard they could get their fill of crumb catches!',
 'explanation': 'This joke is a play on words. In cricket, a "catch" refers to when a fielder catches the ball hit by the batsman, while in a bakery, crumb catches are pastries or bread with a lot of crumbs. So, the cricket team went to the bakery because they heard they could get their fill of crumb catches, meaning they could enjoy some delicious baked goods while also possibly improving their catching skills.'}

In [9]:
workflow.get_state(config2) # for the thread_id=2 this entire data stored in the saparate memory

StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket team go to the bakery?\n\nBecause they heard they could get their fill of crumb catches!', 'explanation': 'This joke is a play on words. In cricket, a "catch" refers to when a fielder catches the ball hit by the batsman, while in a bakery, crumb catches are pastries or bread with a lot of crumbs. So, the cricket team went to the bakery because they heard they could get their fill of crumb catches, meaning they could enjoy some delicious baked goods while also possibly improving their catching skills.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f19bf0e-6edc-69a2-8002-3ebb69b26bd5'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-19T17:10:39.399568+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f19bf0e-5f5a-654d-8001-0585230ce2e9'}}, tasks=(), interrupts=())

In [10]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket team go to the bakery?\n\nBecause they heard they could get their fill of crumb catches!', 'explanation': 'This joke is a play on words. In cricket, a "catch" refers to when a fielder catches the ball hit by the batsman, while in a bakery, crumb catches are pastries or bread with a lot of crumbs. So, the cricket team went to the bakery because they heard they could get their fill of crumb catches, meaning they could enjoy some delicious baked goods while also possibly improving their catching skills.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f19bf0e-6edc-69a2-8002-3ebb69b26bd5'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-19T17:10:39.399568+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f19bf0e-5f5a-654d-8001-0585230ce2e9'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic